# Reusable AI Application Components

Notebooks 04-08 each rebuilt the model, the tools and the agent from scratch. In a real app you build them **once** and import them.

All the calculator pieces now live in [calc_components.py](calc_components.py), next to this notebook:

| Piece | What it is |
|---|---|
| `get_model()` | one place to set the Bedrock model id, temperature, `max_tokens`, timeouts |
| `CALCULATOR_TOOLS` | the four tools: `add`, `subtract`, `multiply`, `divide` |
| `CALCULATOR_PROMPT` | the system prompt |
| `build_calculator_agent()` | assembles model + tools + prompt (+ optional memory, middleware, structured output) |
| `ask()` | sends a question and prints the tool calls and the answer |

Notebooks 10-12 import from this file.


## Setup

In [ ]:
%%capture
%pip install --upgrade langgraph langchain langchain-aws langchain-community
%pip install --upgrade python-dotenv

In [ ]:
from calc_components import (
    CALCULATOR_PROMPT,
    CALCULATOR_TOOLS,
    ask,
    build_calculator_agent,
    get_model,
)


## Default agent: one line

In [ ]:
agent = build_calculator_agent()
ask(agent, "What is 12 plus 8?");


## Same component, different options

Every variation is an argument, not copy-pasted code.

**With memory** (notebook 04):

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = build_calculator_agent(checkpointer=InMemorySaver())

ask(memory_agent, "What is 12 plus 8?", thread_id="demo")
ask(memory_agent, "Multiply that by 5", thread_id="demo");


**With retries and call limits** (notebook 08):

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ModelRetryMiddleware

safe_agent = build_calculator_agent(
    middleware=[
        ModelRetryMiddleware(max_retries=2),
        ModelCallLimitMiddleware(run_limit=6, exit_behavior="end"),
    ],
)
ask(safe_agent, "What is 100 divided by 4?");


**With structured output** (notebook 06):

In [ ]:
from pydantic import BaseModel


class CalcAnswer(BaseModel):
    result: float
    explanation: str


structured_agent = build_calculator_agent(response_format=CalcAnswer)

result = ask(structured_agent, "What is 7 times 6?", verbose=False)
result["structured_response"]


**With a different model or a different prompt:**

In [ ]:
pirate_agent = build_calculator_agent(
    get_model(temperature=0.7, max_tokens=200),
    system_prompt=CALCULATOR_PROMPT + "\nAnswer like a pirate.",
)
ask(pirate_agent, "What is 12 plus 8?");


## Rules of thumb for reusable components

1. **One factory per thing** (`get_model`, `build_..._agent`). Change the model id in one place.
2. **Sensible defaults, everything overridable** with keyword arguments.
3. **No work at import time.** Importing the module must not call the model or read secrets. `get_model()` loads `.env` only when called.
4. **Keep prompts and tools as named constants** so tests and other notebooks can reuse them.
5. **Return data, not just prints.** `ask()` returns the full result so callers can inspect messages and token usage.
6. When the file grows, move it to a package (`my_app/models.py`, `my_app/tools.py`, `my_app/agents.py`).
